## Multi-energy system scheduling

### Simple formulation: ideal grid

In [ ]:
# pip install --upgrade nbstripout
# pip install pyomo gurobipy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyomo.environ as pyo

#### Data gathering

In [ ]:
#------------------------------------------------------------------------------------------------
# PARAMETERS
#------------------------------------------------------------------------------------------------


# LOADS -----------------------------------------------------------------------------------------
Pl_comm_rtd = 1     # Rated commertial load power [MW]
Pl_ind_rtd = 1      # Rated industrial load power [MW]
Pl_resid_rtd = 1    # Rated residential load power [MW]

# PV --------------------------------------------------------------------------------------------
Ppv_rtd = 2    # PV rated power [MW]

# BESS ------------------------------------------------------------------------------------------
Pb_rtd = 1              # BESS rated power [MW]
Pb_ch_max = Pb_rtd      # BESS maximum charging power [MW]
Pb_dch_max = Pb_rtd     # BESS maximum discharging power [MW]
C_rate = 0.5            # BESS C-rate [1/h]
Eb_rtd = Pb_rtd / C_rate    # BESS rated capacity [MWh]
DoD = 0.9               # BESS depth of discharge
SoC_max = 1             # maximum SoC level
SoC_min = 1 - DoD       # minimum SoC level
eta_b_rt = 0.931        # battery round-trip efficiency
eta_b_inv_rt = 0.98     # inverter round-trip efficiency
eta_bch = np.sqrt(eta_b_rt)*np.sqrt(eta_b_inv_rt)       # BESS charging efficiency
eta_bdch = np.sqrt(eta_b_rt)*np.sqrt(eta_b_inv_rt)      # BESS discharging efficiency
sig_bsd = 0.01          # BESS self-discharge rate [1/h] ### NO REFERENCE ###

In [ ]:
#------------------------------------------------------------------------------------------------
# EL MODEL AND PIECEWISE LINEAR APPROXIMATION
#------------------------------------------------------------------------------------------------


# ORIGINAL MODEL --------------------------------------------------------------------------------
# General constants and parameters
T0 = 20 + 273.15    # standard temperature [K]
p0 = 1.01325e5      # standard pressure [Pa]
R = 8.3145          # universal gas constant [J/(mol*K)]
F = 96485           # Faraday constant [C/mol]
z = 2               # number of electrons transferred in electrolysis reaction
Mm = 2.01568e-3     # molar mass of hydrogen [kg/mol]

# Electrolyzer parameters
p = 30e5            # operating pressure [Pa]
T = 70 + 273.15     # operating temperature [K]
R_io = 0.326        # internal stack resistance at standard conditions [Ohm]
k = 0.0395          # fitting parameter
dR_t = -3.812e-3    # temperature coefficient of resistance [Ohm/K]
e_rev0 = 1.476      # reversible voltage at standard conditions [V]
eta_F = 1           # Faraday efficiency

Pel_rtd_W = 1e6       # EL rated power [W]
Pel_min_ratio = 0.1                   # EL minimum power ratio [-]
Pel_min_W = Pel_min_ratio * Pel_rtd_W     # EL minimum power [W]
n_s = 275           # number of cells in series
n_p = 3700          # number of cells in parallel
n_tot = n_s * n_p   # total number of cells

# (Empirical) Electrical model
V_rev = e_rev0 + (R*T)/(2*F) * np.log(p/p0)         # reversible voltage [V]
Ri = R_io + k * np.log(p/p0) + dR_t * (T - T0)      

Pel_fit = np.linspace(Pel_min_W, Pel_rtd_W, 901)  # power array for fitting [W]
Pel_fit_cell = Pel_fit / n_tot           # cell power array for fitting [W]

i_fit = (-V_rev + np.sqrt(V_rev**2 + 4*Ri*Pel_fit_cell)) / (2*Ri)      # current array [A]
H2_prod_fit = eta_F * Mm * 3600 * (n_tot * i_fit) / (z * F)            # H2 production rate array for fitting [kg/h]



# PIECEWISE LINEAR MODEL (3 segments) -----------------------------------------------------------
H2_prod_reduce = H2_prod_fit[::10]
Pel_reduce = Pel_fit[::10]

# Search the optimal breakpoint for the piecewise linear approximation
n_break = 2    # number of breakpoints for the pwl model (x breakpoints => x+1 segments)

opt_slope_s1 = 0
opt_slope_s2 = 0
opt_slope_s3 = 0
opt_intercept_s1 = 0
opt_intercept_s2 = 0
opt_intercept_s3 = 0
abs_err_best = np.inf
pwl_breakpoint = 0

for p1 in range(1, len(Pel_reduce) - n_break):
    slope_s1 = (H2_prod_reduce[p1] - H2_prod_reduce[0]) / (Pel_reduce[p1] - Pel_reduce[0])    # slope of the first segment
    intercept_s1 = H2_prod_reduce[0] - slope_s1 * Pel_reduce[0]                               # intercept of the first segment
    for p2 in range(p1 + 1, len(Pel_reduce) - n_break + 1):
        slope_s2 = (H2_prod_reduce[p2] - H2_prod_reduce[p1]) / (Pel_reduce[p2] - Pel_reduce[p1])  # slope of the second segment
        intercept_s2 = H2_prod_reduce[p1] - slope_s2 * Pel_reduce[p1]
        slope_s3 = (H2_prod_reduce[-1] - H2_prod_reduce[p2]) / (Pel_reduce[-1] - Pel_reduce[p2])  # slope of the third segment
        intercept_s3 = H2_prod_reduce[p2] - slope_s3 * Pel_reduce[p2]
        
        abs_err = sum(abs(H2_prod_fit[0:p1*10] - (slope_s1 * Pel_fit[0:p1*10] + intercept_s1))) + \
          sum(abs(H2_prod_fit[p1*10:p2*10] - (slope_s2 * Pel_fit[p1*10:p2*10] + intercept_s2))) + \
          sum(abs(H2_prod_fit[p2*10:] - (slope_s3 * Pel_fit[p2*10:] + intercept_s3)))
        # print(abs_err)

        if abs_err < abs_err_best:
            abs_err_best = abs_err
            opt_slope_s1 = slope_s1
            opt_slope_s2 = slope_s2
            opt_slope_s3 = slope_s3
            opt_intercept_s1 = intercept_s1
            opt_intercept_s2 = intercept_s2
            opt_intercept_s3 = intercept_s3
            pwl_breakpoint_p1 = p1*10
            pwl_breakpoint_p2 = p2*10
            # print(f"Last best breackpoint indexes = {pwl_breakpoint_p1}, {pwl_breakpoint_p2}")

            
# Print results
print(f"Best breakpoint indexes: {pwl_breakpoint_p1}, {pwl_breakpoint_p2} \n Absolute error = {abs_err_best}")
print(f"They correspond to the power values: {Pel_fit[pwl_breakpoint_p1]} W and {Pel_fit[pwl_breakpoint_p2]} W")
print(f"Slopes and intecepts of the segments are are: \
      \n First segment slope = {opt_slope_s1}, intercept = {opt_intercept_s1}) \
      \n Second segment slope = {opt_slope_s2}, intercept = {opt_intercept_s2} \
      \n Third segment slope = {opt_slope_s3}, intercept = {opt_intercept_s3}")

# H2 production according to the pwl model (for plotting)
H2_prod_pwl_s1 = opt_slope_s1 * Pel_fit[0:pwl_breakpoint_p1] + opt_intercept_s1
H2_prod_pwl_s2 = opt_slope_s2 * Pel_fit[pwl_breakpoint_p1:pwl_breakpoint_p2] + opt_intercept_s2
H2_prod_pwl_s3 = opt_slope_s3 * Pel_fit[pwl_breakpoint_p2:] + opt_intercept_s3
H2_prod_pwl = np.concatenate((H2_prod_pwl_s1, H2_prod_pwl_s2, H2_prod_pwl_s3))


# Adjust parameters to express EL power in MW
Pel_rtd = 1                         # EL rated power [MW] (also maximum power)
Pel_min = Pel_min_ratio * Pel_rtd   # EL minimum turndown level [MW] (also minimum power level)
opt_slope_s1 = opt_slope_s1 * 1e6
opt_slope_s2 = opt_slope_s1 * 1e6
opt_slope_s3 = opt_slope_s3 * 1e6

In [ ]:
#------------------------------------------------------------------------------------------------
# PROFILES
#------------------------------------------------------------------------------------------------

# Reading the Excel files
C_ele_original = pd.read_excel('GME_20240101_20241231_MGP_PrezziZonali_CSUD.xlsx', decimal=',')
# print(C_ele_original)
Pl_Ppv_Pwt_original = pd.read_excel('Dati ENEA.xlsx')
# print(Pl_Ppv_Pwt_original)


# Extraction of useful profiles
C_ele_full = C_ele_original['€/MWh'].tolist()               # Zonal price (CSUD 2024) [€/MWh], from 01/01/24 to 31/12/24 (leap year, 366 days)
C_ele = np.array(C_ele_full[:1416] + C_ele_full[1440:])     # Removing 29th Feb data to have 365 days (8760 hours) and turning into ndarray

Pl_comm_to_arrange = Pl_Ppv_Pwt_original['Commerciale'].tolist()            # Commercial load profile [p.u.], from 16/10/20 to 15/10/21
Pl_comm = np.array(Pl_comm_to_arrange[1848:] + Pl_comm_to_arrange[:1848])   # Rearranged data from 01/01/21 to 15/10/21 and then from 16/10/20 to 31/12/20

Pl_ind_to_arrange = Pl_Ppv_Pwt_original['Industriale'].tolist()             # Industrial load profile [p.u.], from 16/10/20 to 15/10/21
Pl_ind = np.array(Pl_ind_to_arrange[1848:] + Pl_ind_to_arrange[:1848])

Pl_resid_to_arrange = Pl_Ppv_Pwt_original['Residenziale'].tolist()          # Residential load profile [p.u.], from 16/10/20 to 15/10/21
Pl_resid = np.array(Pl_resid_to_arrange[1848:] + Pl_resid_to_arrange[:1848])

Ppv_to_arrange = Pl_Ppv_Pwt_original['PV'].tolist()         # PV generation profile [??], from 16/10/20 to 15/10/21
Ppv = np.array(Ppv_to_arrange[1848:] + Ppv_to_arrange[:1848])

Pwt_to_arrange = Pl_Ppv_Pwt_original['Wind'].tolist()       # WT generation profile [??], from 16/10/20 to 15/10/21
Pwt = np.array(Pwt_to_arrange[1848:] + Pwt_to_arrange[:1848])


# Checking the data
time = np.arange(0, 8760)

# Plot settings
plt.rc('figure', figsize=(20, 20))
plt.rc('font', family='monospace', weight='bold', size=11)
plt.rc('axes', labelsize=11, titlesize=14)
plt.rc('legend', fontsize=11)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)
plt.rcParams['axes.grid'] = True
plt.rcParams['lines.linewidth'] = 2

# Plots
fig, axs = plt.subplots(6, 1, layout='constrained')
axs[0].plot(time, C_ele, color='blue')
axs[0].set_xlabel('x label')
axs[0].set_ylabel('y label')
axs[0].set_title("Electricity Price [€/MWh]")
axs[0].legend()

axs[1].plot(time, Pl_comm)
axs[1].set_title("Commercial load [p.u.]")
axs[2].plot(time, Pl_ind)
axs[2].set_title("Industrial load [p.u.]")
axs[3].plot(time, Pl_resid)
axs[3].set_title("Residential load [p.u.]")
axs[4].plot(time, Ppv)
axs[4].set_title("PV generation [??]")
axs[5].plot(time, Pwt)
axs[5].set_title("WT generation [??]")

In [ ]:
# Profiles adjustments based on rated powers
Pl_comm = Pl_comm * Pl_comm_rtd         # Commercial load profile [MW]
Pl_ind = Pl_ind * Pl_ind_rtd            # Industrial load profile [MW]
Pl_resid = Pl_resid * Pl_resid_rtd      # Residential load profile [MW]
Ppv = Ppv / 2 * Ppv_rtd                 # PV generation profile [MW]
# Pwt = Pwt * Pwt_rtd                   # WT generation profile [MW]

# Checking the data after adjustments
fig, axs = plt.subplots(6, 1, layout='constrained')
axs[0].plot(time, C_ele, color='blue')
axs[0].set_xlabel('x label')
axs[0].set_ylabel('y label')
axs[0].set_title("Electricity Price [€/MWh]")
axs[0].legend()

axs[1].plot(time, Pl_comm)
axs[1].set_title("Commercial load [MW]")
axs[2].plot(time, Pl_ind)
axs[2].set_title("Industrial load [MW]")
axs[3].plot(time, Pl_resid)
axs[3].set_title("Residential load [MW]")
axs[4].plot(time, Ppv)
axs[4].set_title("PV generation [MW]")
# axs[5].plot(time, Pwt)
# axs[5].set_title("WT generation")

In [ ]:
#------------------------------------------------------------------------------------------------
# PROBLEM DEFINITION IN PYOMO
#------------------------------------------------------------------------------------------------


# GENERAL PROBLEM PARAMETERS
time_window = 8760      # time window [h]

# Model -----------------------------------------------------------------------------------------
m = pyo.ConcreteModel()

# Sets ------------------------------------------------------------------------------------------
m.TIME = pyo.RangeSet(0, time_window, 1)              # Set of the timesteps
m.GEN = pyo.Set(initialize = ['BESS_dch', 'FC'])      # Set of the controllable generating units
### QUESTION: does the slack bus need a variable?
m.ABS = pyo.Set(initialize = ['BESS_ch', 'EL'])       # Set of the controllable load/power-absorbing units


